In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time
import logging
import datetime

In [ ]:
HEADERS = {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/120.0.0.0 Safari/537.36","Accept-Language": "ru-RU,ru;q=0.9",}
YEARS = ["2026", "2025", "2024", "2023", "2022", "2021","2020", "2019", "2018", "2017", "2016", "2015","2014", "2013", "2012", "2011", "2010", "2009","2008", "2007", "2006", "2005", "2004", "2003","2002", "2001", "2000"]

https://www.zenrows.com/blog/user-agent-web-scraping#what-is-user-agent

In [ ]:
def scrapim(url):
    page = requests.get(url, headers=HEADERS, timeout=15)
    if page.status_code != 200:
      return []
    soup = BeautifulSoup(page.text, "html.parser")
    seen = set()
    result = []

    for card in soup.find_all("div", class_=lambda x: x and "redesign_afisha_movie" in x and "main" not in x):
        t = card.find("a", class_="redesign_afisha_movie_main_title")
        s = t.find("strong") if t else None
        title_ru = s.get_text(strip=True) if s else (t.get_text(strip=True) if t else None)

        en = card.find("div", class_="redesign_afisha_movie_main_subtitle")
        sp = en.find("span") if en else None
        title_en = sp.get_text(strip=True) if sp else (en.get_text(strip=True) if en else None)

        a = card.find("a", href=True)
        link = "https://www.film.ru" + a["href"] if a else None

        if link in seen:
            continue
        seen.add(link)

        rt = card.find("div", class_="redesign_afisha_movie_main_rating")
        rt = rt.text.strip() if rt else ""

        parts_imdb = rt.split("IMDb:")
        parts_users = parts_imdb[0].split("зрители:")
        parts_filmru = parts_users[0].split("film.ru:")

        if title_ru:
            result.append({
                "title_ru": title_ru,
                "title_en": title_en,
                "rating_filmru": parts_filmru[1].strip() if len(parts_filmru) > 1 else None,
                "rating_users": parts_users[1].strip() if len(parts_users) > 1 else None,
                "rating_imdb": parts_imdb[1].strip() if len(parts_imdb) > 1 else None,
                "url": link })

    return result

In [ ]:
all_films = []
seen_urls_global = set()

for yr in YEARS:
    page = 1
    movies_do = len(all_films)
    while True:
        url = f"https://www.film.ru/a-z/movies/{yr}" if page == 1 else f"https://www.film.ru/a-z/movies/{yr}?page={page}"
        batch = scrapim(url)
        fresh = [m for m in batch if m["url"] not in seen_urls_global]
        for m in fresh:
            seen_urls_global.add(m["url"])
        if not fresh:
            break

        all_films.extend(fresh)
        print(yr, "страница", page, "---", len(all_films))
        if len(all_films) % 500 == 0:
            pd.DataFrame(all_films).to_csv("filmru_movies.csv", index=False, encoding="utf-8-sig")
            print("автосохранение", len(all_films))

        if len(all_films) - movies_do >= 500:
            print(yr,"500 собрали")
            break
        page += 1
        time.sleep(0.5)

print("Всего", len(all_films))

In [ ]:
df_filmru = pd.DataFrame(all_films)
print(df_filmru.shape)
df_filmru.head(3)

(30, 6)


,title_ru,title_en,rating_filmru,rating_users,rating_imdb,url
0,Жозефина,Josephine,10,5.5,8,https://www.film.ru/movies/zhozefina
1,28 лет спустя: Храм костей,28 Years Later: The Bone Temple,9,5.9,7.3,https://www.film.ru/movies/28-let-spustya-hram...
2,"День, когда она вернулась",Geunyeoga doraon nal,9,5.8,5.6,https://www.film.ru/movies/den-kogda-ona-vernulas


In [ ]:
df_filmru.to_csv("filmru_movies.csv", index=False, encoding="utf-8-sig")